In [0]:
%pip install yfinance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 57.0 MB/s eta 0:00:00
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Not uninstalling cffi at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-918659fe-cc66-4de6-947e-44b71d6be897
    Can't uninstall 'cffi'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import yfinance as yf
import pandas as pd
from pyspark.sql.functions import col

# Same 50 tickers as original pipeline
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "AMD",
    "JPM", "GS", "BAC", "MS", "BLK", "C", "WFC", "AXP",
    "JNJ", "UNH", "PFE", "MRK", "ABT", "TMO", "LLY", "ABBV",
    "XOM", "CVX", "COP", "SLB", "OXY", "NEE",
    "WMT", "PG", "KO", "PEP", "COST", "NKE", "MCD", "SBUX",
    "CAT", "HON", "UPS", "BA", "GE", "LMT", "RTX", "DE",
    "DIS", "NFLX", "V", "MA"
]

# Step 1: Find the last date already in the table
last_date = spark.sql("SELECT MAX(Date) as max_date FROM bronze_stock_prices").collect()[0]["max_date"]
print(f"Last date in database: {last_date}")

# Step 2: Pull only NEW data (from last date to today)
start_date = (pd.to_datetime(last_date) + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
print(f"Fetching data from: {start_date}")

data = yf.download(tickers, start=start_date, group_by="ticker")

if data.empty:
    print("No new data available. Market may be closed or data already up to date.")
else:
    # Step 3: Reshape to long format (same as original notebook)
    records = []
    for ticker in tickers:
        try:
            df = data[ticker].copy()
            df["Ticker"] = ticker
            df["Date"] = df.index
            records.append(df)
        except:
            print(f"Skipping {ticker}")

    new_data = pd.concat(records).reset_index(drop=True)
    new_data.columns = ["Open", "High", "Low", "Close", "Volume", "Ticker", "Date"]
    new_data = new_data[["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"]]
    new_data = new_data.dropna(subset=["Close"])

    print(f"New rows to add: {len(new_data)}")

    # Step 4: Append new data to existing table
    spark_df = spark.createDataFrame(new_data)
    spark_df.write.mode("append").saveAsTable("bronze_stock_prices")

    print("New data appended to bronze_stock_prices!")

Last date in database: 2026-05-29 00:00:00
Fetching data from: 2026-05-30


[*********************100%***********************]  50 of 50 completed


New rows to add: 850
New data appended to stock_prices_raw!


In [0]:
# Step 5: Recalculate stock_analytics (daily returns + sectors)
spark.sql("""
    CREATE OR REPLACE TABLE silver_stock_analytics AS
    WITH daily_returns AS (
        SELECT 
            p.*,
            s.Sector,
            LAG(p.Close) OVER (PARTITION BY p.Ticker ORDER BY p.Date) AS prev_close,
            ROUND(((p.Close - LAG(p.Close) OVER (PARTITION BY p.Ticker ORDER BY p.Date)) 
                   / LAG(p.Close) OVER (PARTITION BY p.Ticker ORDER BY p.Date)) * 100, 4) AS daily_return_pct
        FROM bronze_stock_prices p
        JOIN sector_mapping s ON p.Ticker = s.Ticker
    )
    SELECT * FROM daily_returns
    WHERE prev_close IS NOT NULL
""")

# Step 6: Recalculate risk_metrics
spark.sql("""
    CREATE OR REPLACE TABLE gold_risk_metrics AS
    WITH market_avg AS (
        SELECT Date, AVG(daily_return_pct) AS market_return
        FROM silver_stock_analytics
        GROUP BY Date
    )
    SELECT 
        a.Ticker,
        a.Sector,
        ROUND(AVG(a.daily_return_pct), 4) AS avg_daily_return,
        ROUND(STDDEV(a.daily_return_pct), 4) AS daily_volatility,
        COUNT(*) AS trading_days,
        ROUND(MIN(a.Close), 2) AS min_price,
        ROUND(MAX(a.Close), 2) AS max_price,
        ROUND(AVG(a.daily_return_pct) / NULLIF(STDDEV(a.daily_return_pct), 0), 4) AS sharpe_proxy,
        ROUND(
            COVAR_SAMP(a.daily_return_pct, m.market_return) 
            / NULLIF(VAR_SAMP(m.market_return), 0), 4
        ) AS beta,
        ROUND(PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY a.daily_return_pct), 4) AS var_95
    FROM silver_stock_analytics a
    JOIN market_avg m ON a.Date = m.Date
    GROUP BY a.Ticker, a.Sector
""")

# Step 7: Recalculate sector_performance
spark.sql("""
    CREATE OR REPLACE TABLE gold_sector_performance AS
    SELECT 
        Sector,
        COUNT(DISTINCT Ticker) AS num_stocks,
        ROUND(AVG(avg_daily_return), 4) AS sector_avg_return,
        ROUND(AVG(daily_volatility), 4) AS sector_avg_volatility,
        ROUND(AVG(sharpe_proxy), 4) AS sector_avg_sharpe,
        ROUND(AVG(beta), 4) AS sector_avg_beta,
        ROUND(AVG(var_95), 4) AS sector_avg_var
    FROM gold_risk_metrics
    GROUP BY Sector
""")

# Verify
total = spark.sql("SELECT COUNT(*) as rows FROM bronze_stock_prices").collect()[0]["rows"]
latest = spark.sql("SELECT MAX(Date) as latest FROM bronze_stock_prices").collect()[0]["latest"]
print(f"Total rows in database: {total}")
print(f"Latest date: {latest}")
print("All tables refreshed!")

Total rows in database: 63600
Latest date: 2026-06-24 00:00:00
All tables refreshed!


In [0]:
# DATA QUALITY CHECKS
print("=" * 50)
print("DATA QUALITY REPORT")
print("=" * 50)

# Check 1: Any null prices?
nulls = spark.sql("""
    SELECT COUNT(*) as null_count 
    FROM stock_prices_raw 
    WHERE Close IS NULL OR Open IS NULL
""").collect()[0]["null_count"]
print(f"\n1. Null prices: {nulls} {'✓ PASS' if nulls == 0 else '✗ FAIL'}")

# Check 2: Any duplicate rows (same ticker + same date)?
dupes = spark.sql("""
    SELECT COUNT(*) as dupe_count FROM (
        SELECT Ticker, Date, COUNT(*) as cnt 
        FROM stock_prices_raw 
        GROUP BY Ticker, Date 
        HAVING cnt > 1
    )
""").collect()[0]["dupe_count"]
print(f"2. Duplicate rows: {dupes} {'✓ PASS' if dupes == 0 else '✗ FAIL'}")

# Check 3: All 50 tickers present?
ticker_count = spark.sql("""
    SELECT COUNT(DISTINCT Ticker) as tickers 
    FROM stock_prices_raw
""").collect()[0]["tickers"]
print(f"3. Unique tickers: {ticker_count} {'✓ PASS' if ticker_count == 50 else '✗ FAIL'}")

# Check 4: Any negative prices?
neg_prices = spark.sql("""
    SELECT COUNT(*) as neg_count 
    FROM stock_prices_raw 
    WHERE Close < 0 OR Open < 0 OR High < 0 OR Low < 0
""").collect()[0]["neg_count"]
print(f"4. Negative prices: {neg_prices} {'✓ PASS' if neg_prices == 0 else '✗ FAIL'}")

# Check 5: Any missing dates (gaps in trading days)?
date_count = spark.sql("""
    SELECT Ticker, COUNT(DISTINCT Date) as days 
    FROM stock_prices_raw 
    GROUP BY Ticker 
    ORDER BY days ASC 
    LIMIT 5
""").toPandas()
print(f"\n5. Tickers with fewest trading days:")
print(date_count.to_string(index=False))

# Check 6: Latest date per ticker (catch any stale tickers)
stale = spark.sql("""
    SELECT Ticker, MAX(Date) as latest_date 
    FROM stock_prices_raw 
    GROUP BY Ticker 
    HAVING MAX(Date) < (SELECT MAX(Date) FROM stock_prices_raw)
""").toPandas()
print(f"\n6. Stale tickers (not updated to latest date): {len(stale)} {'✓ PASS' if len(stale) == 0 else '✗ WARNING'}")
if len(stale) > 0:
    print(stale.to_string(index=False))

print("\n" + "=" * 50)
print("QUALITY CHECK COMPLETE")
print("=" * 50)

DATA QUALITY REPORT

1. Null prices: 0 ✓ PASS
2. Duplicate rows: 0 ✓ PASS
3. Unique tickers: 50 ✓ PASS
4. Negative prices: 0 ✓ PASS

5. Tickers with fewest trading days:
Ticker  days
  AMZN  1272
  MSFT  1272
  AAPL  1272
 GOOGL  1272
  NVDA  1272

6. Stale tickers (not updated to latest date): 0 ✓ PASS

QUALITY CHECK COMPLETE


In [0]:
# RENAME TABLES TO MEDALLION ARCHITECTURE
# Bronze = raw data, Silver = cleaned + transformed, Gold = business-ready metrics

# Bronze layer
spark.sql("ALTER TABLE stock_prices_raw RENAME TO bronze_stock_prices")

# Silver layer
spark.sql("ALTER TABLE stock_analytics RENAME TO silver_stock_analytics")

# Gold layer
spark.sql("ALTER TABLE risk_metrics RENAME TO gold_risk_metrics")
spark.sql("ALTER TABLE sector_performance RENAME TO gold_sector_performance")

# sector_mapping stays as a reference table
print("Tables renamed to medallion architecture!")
print()

# Verify
tables = spark.sql("SHOW TABLES").toPandas()
print(tables[["tableName"]].to_string(index=False))

Tables renamed to medallion architecture!

              tableName
    bronze_stock_prices
      gold_risk_metrics
gold_sector_performance
         sector_mapping
 silver_stock_analytics
